# ROADGUARD — YOLOv8s Training on Kaggle (Dual T4 GPU)
### Indian Vehicle License Plate Detection (7,233 Annotated Images)

**Hardware**: Select **Accelerator ➔ GPU T4 x2** in the right-hand Kaggle sidebar.
**Input**: Upload `unique_dataset_combined.zip` via **Add Input ➔ Upload**.

In [ ]:
# 1. Install Ultralytics
!pip install -q ultralytics
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Locate and Copy Dataset to /kaggle/working/ (Writable Directory)
import os, glob, shutil, zipfile
from pathlib import Path

# Search for train/images anywhere in /kaggle/input
print("Searching for dataset files in /kaggle/input...")
train_matches = glob.glob("/kaggle/input/**/train/images", recursive=True)
zip_matches = glob.glob("/kaggle/input/**/*.zip", recursive=True)

dest_root = Path("/kaggle/working/unique dataset")
dest_root.mkdir(parents=True, exist_ok=True)

if train_matches:
    # Dataset is already unzipped by Kaggle into /kaggle/input/
    source_root = Path(train_matches[0]).parent.parent
    print(f"Found extracted dataset at: {source_root}")
    for folder in ['train', 'valid', 'test']:
        src = source_root / folder
        dst = dest_root / folder
        if src.exists() and not dst.exists():
            print(f"Copying {folder} to writable working directory...")
            shutil.copytree(src, dst, dirs_exist_ok=True)
elif zip_matches:
    print(f"Unzipping {zip_matches[0]}...")
    with zipfile.ZipFile(zip_matches[0], 'r') as z:
        z.extractall("/kaggle/working")
else:
    print("Searching all directories in /kaggle/input:")
    for root, dirs, files in os.walk("/kaggle/input"):
        print(root, dirs[:3], len(files))

# Write data.yaml with absolute paths for Kaggle
yaml_text = """
path: /kaggle/working/unique dataset
train: train/images
val: valid/images
test: test/images

nc: 1
names: ['LicensePlate']
"""
with open(dest_root / "data.yaml", "w") as f:
    f.write(yaml_text.strip())

train_c = len(list((dest_root / 'train' / 'images').glob('*.*')))
val_c = len(list((dest_root / 'valid' / 'images').glob('*.*')))
test_c = len(list((dest_root / 'test' / 'images').glob('*.*')))

print("\n--- DATASET VERIFICATION ---")
print(f"Train images: {train_c}")
print(f"Val images:   {val_c}")
print(f"Test images:  {test_c}")

In [ ]:
# 3. Train YOLOv8s (Small) on 7,233 Images for 50 Epochs
from ultralytics import YOLO

# Load YOLOv8 Small architecture
model = YOLO("yolov8s.pt")

# Train with AdamW, Cosine Annealing, and Early Stopping
results = model.train(
    data="/kaggle/working/unique dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=10,
    device=0,
    project="/kaggle/working/ROADGUARD_Plate_Detector",
    name="yolov8s_combined_detector",
    exist_ok=True
)

In [ ]:
# 4. Evaluate Model on 605 Unseen Test Images
metrics = model.val(split="test")
print("\n=== FINAL TEST EVALUATION RESULTS ===")
print(f"mAP@0.50:        {metrics.box.map50 * 100:.2f}%")
print(f"mAP@0.50:0.95:   {metrics.box.map * 100:.2f}%")
print(f"Precision (P):   {metrics.box.p[0] * 100:.2f}%")
print(f"Recall (R):      {metrics.box.r[0] * 100:.2f}%")

In [ ]:
# 5. Display Training Loss and Confusion Matrix
from IPython.display import Image, display
from pathlib import Path

results_png = Path("/kaggle/working/ROADGUARD_Plate_Detector/yolov8s_combined_detector/results.png")
if results_png.exists():
    print("Training Convergence Metrics:")
    display(Image(filename=str(results_png)))

cm_png = Path("/kaggle/working/ROADGUARD_Plate_Detector/yolov8s_combined_detector/confusion_matrix.png")
if cm_png.exists():
    print("Confusion Matrix:")
    display(Image(filename=str(cm_png)))

In [ ]:
# 6. Copy weights directly to /kaggle/working/ for 1-Click Download
import shutil
best_src = "/kaggle/working/ROADGUARD_Plate_Detector/yolov8s_combined_detector/weights/best.pt"
if os.path.exists(best_src):
    shutil.copy(best_src, "/kaggle/working/best.pt")
    print(" SUCCESS: 'best.pt' copied to /kaggle/working/best.pt")
    print("You can now download it directly from the Output panel on the right sidebar!")